# Convert ItemsList Exports to Excel

Run the cells in order. The notebook scans `.txt`, `.tsv`, and `.tab` files in this folder, identifies GOBI exports by their required headers, and converts eligible files into Excel workbooks using its built-in 15-column schema. File names do not matter.

In [4]:
import csv
from pathlib import Path

from openpyxl import Workbook

REFERENCE_HEADERS = [
    "Title", "Author", "Editor", "Pub_Year", "ISBN", "LCCN",
    "LC/NLM/Dewey_Class", "Subaccount", "Fund_Code", "PO_Number", "Supplier",
    "Purchase_Option", "Quantity", "Initials", "YBP_Order_Key",
]

WORKING_DIRECTORY = Path.cwd()
print(f"Working directory: {WORKING_DIRECTORY}")

Working directory: /Users/ng3110/Documents/tcPressEbooks


In [5]:
def read_tsv_rows(source_path: Path) -> list[list[str]]:
    """Read a tab-delimited export and remove invalid NUL characters."""
    with source_path.open("r", encoding="utf-8-sig", newline="") as source_file:
        return list(csv.reader((line.replace("\0", "") for line in source_file), delimiter="\t"))


def missing_required_columns(headers: list[str]) -> list[str]:
    """Return the required output fields absent from an export header row."""
    available_headers = set(headers)
    return [header for header in REFERENCE_HEADERS if header not in available_headers]


def convert_file(source_path: Path, output_path: Path) -> int:
    """Convert one tab-delimited export using the built-in Excel schema."""
    rows = read_tsv_rows(source_path)
    if not rows:
        raise ValueError(f"The input file is empty: {source_path.name}")

    source_headers, data_rows = rows[0], rows[1:]
    missing_headers = missing_required_columns(source_headers)
    if missing_headers:
        raise ValueError(
            f"Cannot convert {source_path.name}. Missing required columns: "
            f"{', '.join(missing_headers)}"
        )

    source_indexes = {header: index for index, header in enumerate(source_headers)}
    workbook = Workbook()
    worksheet = workbook.active
    worksheet.title = "20210901_20220831_GOBI_test"
    worksheet.append(REFERENCE_HEADERS)

    for row in data_rows:
        worksheet.append([
            row[source_indexes[header]] if source_indexes[header] < len(row) else ""
            for header in REFERENCE_HEADERS
        ])

    workbook.save(output_path)
    return len(data_rows)

In [6]:
SUPPORTED_EXTENSIONS = {".txt", ".tsv", ".tab"}


def find_gobi_exports(directory: Path) -> list[Path]:
    """Find valid exports by required headers instead of their file names."""
    source_paths = sorted(
        path for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )
    valid_paths = []

    for source_path in source_paths:
        try:
            rows = read_tsv_rows(source_path)
        except UnicodeDecodeError:
            print(f"Skipped {source_path.name}: file is not UTF-8 text.")
            continue

        if not rows or not rows[0]:
            print(f"Skipped {source_path.name}: file is empty or has no header row.")
            continue

        missing_headers = missing_required_columns(rows[0])
        if missing_headers:
            if "Title" in rows[0] or "ISBN" in rows[0] or "YBP_Order_Key" in rows[0]:
                print(
                    f"Skipped {source_path.name}: missing required columns: "
                    f"{', '.join(missing_headers)}"
                )
            continue

        valid_paths.append(source_path)

    return valid_paths


input_paths = find_gobi_exports(WORKING_DIRECTORY)
if not input_paths:
    required_columns = ", ".join(REFERENCE_HEADERS)
    raise FileNotFoundError(
        "No valid GOBI exports found. Required columns: " + required_columns
    )

for source_path in input_paths:
    output_path = source_path.with_suffix(".xlsx")
    if output_path.exists():
        print(f"Skipped {source_path.name}: {output_path.name} already exists.")
        continue

    row_count = convert_file(source_path, output_path)
    print(f"Created {output_path.name} ({row_count} data rows)")

FileNotFoundError: No valid GOBI exports found. Required columns: Title, Author, Editor, Pub_Year, ISBN, LCCN, LC/NLM/Dewey_Class, Subaccount, Fund_Code, PO_Number, Supplier, Purchase_Option, Quantity, Initials, YBP_Order_Key